<a href="https://colab.research.google.com/github/chrliem/bank-telemarketing-optimization/blob/main/Bank_Telemarketing_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Bank Telemarketing Optimization**

This project analyzes a dataset related with direct marketing campaigns of a Portuguese banking institution. The marketing campaigns were based on phone calls. Often, more than one contact to the same client was required, in order to access if the product (bank term deposit) would be ('yes') or not ('no') subscribed.

Dataset source: UCI Machine Learning Repo [Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing)

**Problem Statement:**
Telemarketing campaigns often suffers from low conversion rates and high operational costs due to uninterested leads. Contacting every customer in the database also might be not an effective approce with the limited resources. There is a need to identify the profile of custommer who likely to subscribe to the product.


**Objective:**
To predict whether a client will subscribe to a term deposit. By identifying several indicators, this project aims to:
- Increase conversion rate, by focusing marketing efforts on high probability lead.
- Optimize resource allocation, reduce the number of unnecessary calls or low probability client, so it can lowering the operational cost.
- Understand which factors that most significantly influence client decision

In [66]:
!pip install ucimlrepo

In [67]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import matplotlib.pyplot as plt
from plotly.subplots import make_subplots
from ucimlrepo import fetch_ucirepo
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

In [68]:
bank_marketing = fetch_ucirepo(id=222)

X = bank_marketing.data.features
y = bank_marketing.data.targets

df = pd.concat([X, y], axis=1)

df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN,no
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN,no
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN,no
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN,no


## **Data Overview**

In [69]:
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
df.info()

Rows: 45211, Columns: 17
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   age          45211 non-null  int64 
 1   job          44923 non-null  object
 2   marital      45211 non-null  object
 3   education    43354 non-null  object
 4   default      45211 non-null  object
 5   balance      45211 non-null  int64 
 6   housing      45211 non-null  object
 7   loan         45211 non-null  object
 8   contact      32191 non-null  object
 9   day_of_week  45211 non-null  int64 
 10  month        45211 non-null  object
 11  duration     45211 non-null  int64 
 12  campaign     45211 non-null  int64 
 13  pdays        45211 non-null  int64 
 14  previous     45211 non-null  int64 
 15  poutcome     8252 non-null   object
 16  y            45211 non-null  object
dtypes: int64(7), object(10)
memory usage: 5.9+ MB


In [70]:
display(df.describe())

,age,balance,day_of_week,duration,campaign,pdays,previous
count,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000
mean,40.936210,1362.272058,15.806419,258.163080,2.763841,40.197828,0.580323
std,10.618762,3044.765829,8.322476,257.527812,3.098021,100.128746,2.303441
min,18.000000,-8019.000000,1.000000,0.000000,1.000000,-1.000000,0.000000
25%,33.000000,72.000000,8.000000,103.000000,1.000000,-1.000000,0.000000
50%,39.000000,448.000000,16.000000,180.000000,2.000000,-1.000000,0.000000
75%,48.000000,1428.000000,21.000000,319.000000,3.000000,-1.000000,0.000000
max,95.000000,102127.000000,31.000000,4918.000000,63.000000,871.000000,275.000000


## **Data Quality Assesment**

In [71]:
print(f"Total duplicate rows: {df.duplicated().sum()}")

missing_percent = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Missing %': missing_percent})

print("\nFeatures with Missing Values:")
display(missing_df[missing_df['Missing %'] > 0].sort_values(by='Missing %', ascending=False))

Total duplicate rows: 0

Features with Missing Values:


,Missing %
poutcome,81.747805
contact,28.798301
education,4.107407
job,0.637013


In [72]:
print(f"Range of 'day_of_week': {df['day_of_week'].min()} to {df['day_of_week'].max()}")

not_contacted = (df['pdays'] == -1).sum()
print(f"Clients never contacted before (pdays = -1): {not_contacted} rows ({not_contacted/len(df)*100:.2f}%)")

negative_balance = (df['balance'] < 0).sum()
print(f"Clients with negative balance: {negative_balance} rows")

Q1 = df['campaign'].quantile(0.25)
Q3 = df['campaign'].quantile(0.75)
IQR = Q3 - Q1

upper_bound_campaign = Q3 + 1.5 * IQR

outliers_count = (df['campaign'] > upper_bound_campaign).sum()
print(f"Total outlier rows: {outliers_count} rows ({outliers_count/len(df)*100:.2f}%)")

Range of 'day_of_week': 1 to 31
Clients never contacted before (pdays = -1): 36954 rows (81.74%)
Clients with negative balance: 3766 rows
Total outlier rows: 3064 rows (6.78%)


## **Preprocessing and Feature Engineering**

In [73]:
df.rename(columns={'day_of_week': 'day_of_month'}, inplace=True)

df['poutcome'] = df['poutcome'].fillna('non_existent')

for col in ['contact', 'education', 'job']:
    df[col] = df[col].fillna('unknown')

print(df.isnull().sum()[df.isnull().sum() > 0])

Series([], dtype: int64)


In [74]:
Q1 = df['campaign'].quantile(0.25)
Q3 = df['campaign'].quantile(0.75)
IQR = Q3 - Q1
upper_bound_campaign = Q3 + 1.5 * IQR

df['campaign'] = np.where(df['campaign'] > upper_bound_campaign, upper_bound_campaign, df['campaign'])

print(f"Upper bound campaign: {upper_bound_campaign}")
print(f"Maximum campaign current: {df['campaign'].max()}")

Upper bound campaign: 6.0
Maximum campaign current: 6.0


## **EDA**

### **Overall Campaign Conversion Rate**

In [75]:
target_counts = df['y'].value_counts().reset_index()
target_counts.columns = ['Subscription', 'Count']

fig1 = px.pie(target_counts, values='Count', names='Subscription',
             title='Overall Campaign Conversion Rate',
             hole=0.4,
             color='Subscription',
             color_discrete_map={'yes':'#00CC96', 'no':'#EF553B'})

fig1.update_traces(textposition='inside', textinfo='percent+label')
fig1.show()

### **Account Balance (Annual) Distribution by Subcription Status**

In [76]:
fig2 = px.box(df, x='y', y='balance', color='y',
             title='Account Balance Distribution by Subscription Status',
             labels={'y': 'Subscribed to Term Deposit', 'balance': 'Account Balance (EUR)'},
             color_discrete_map={'yes':'#00CC96', 'no':'#EF553B'})
fig2.update_layout(yaxis=dict(range=[-1000, 5000]))
fig2.show()

### **Conversion Rate based on Number of Calls Attempt**

In [77]:
conversion_by_campaign = df.groupby('campaign')['y'].apply(lambda x: (x == 'yes').mean() * 100).reset_index()
conversion_by_campaign.rename(columns={'y': 'Conversion Rate (%)'}, inplace=True)

fig3 = px.bar(conversion_by_campaign, x='campaign', y='Conversion Rate (%)',
             title='Conversion Rate vs. Number of Calls Made in This Campaign',
             labels={'campaign': 'Number of Calls Made'},
             text='Conversion Rate (%)')

fig3.update_traces(texttemplate='%{text:.1f}%', textposition='outside', marker_color='#636EFA')

fig3.update_layout(xaxis=dict(tickmode='linear', dtick=1))
fig3.show()

### **Conversion Rate by Job Category**

In [78]:
job_conv = df.groupby('job')['y'].apply(lambda x: (x == 'yes').mean() * 100).reset_index()
job_conv = job_conv.sort_values(by='y', ascending=False)

fig_job = px.bar(job_conv, x='job', y='y',
                 title='Conversion Rate by Job Category',
                 text=job_conv['y'].round(1),
                 labels={'job': 'Occupation', 'y': 'Conversion Rate (%)'},
                 color='y', color_continuous_scale='Viridis')
fig_job.show()

### **Conversion Rate based on Client's Loan**

In [79]:
h_conv = df.groupby('housing')['y'].apply(lambda x: (x == 'yes').mean() * 100).reset_index()
l_conv = df.groupby('loan')['y'].apply(lambda x: (x == 'yes').mean() * 100).reset_index()

fig = make_subplots(rows=1, cols=2, subplot_titles=('Housing Loan', 'Personal Loan'))

fig.add_trace(
    go.Bar(x=h_conv['housing'], y=h_conv['y'], name='Housing',
           marker_color=['#EF553B' if x=='no' else '#00CC96' for x in h_conv['housing']],
           text=h_conv['y'].round(1), textposition='auto'),
    row=1, col=1
)

fig.add_trace(
    go.Bar(x=l_conv['loan'], y=l_conv['y'], name='Personal Loan',
           marker_color=['#EF553B' if x=='no' else '#00CC96' for x in l_conv['loan']],
           text=l_conv['y'].round(1), textposition='auto'),
    row=1, col=2
)

fig.update_layout(title_text='Conversion Rate Comparison: Housing Load and Personal Load', showlegend=False)
fig.show()

### **Conversion Rate by Education LEvel**

In [80]:
edu_conv = df.groupby('education')['y'].apply(lambda x: (x == 'yes').mean() * 100).reset_index()
fig_edu = px.bar(edu_conv, x='education', y='y',
                 title='Conversion Rate by Education Level',
                 labels={'education': 'Education Level', 'y': 'Conversion Rate (%)'},
                 text=edu_conv['y'].round(1))
fig_edu.show()

### **Client Age Distribution and Subcription Status**

In [81]:
fig_age = px.histogram(df, x='age', color='y', marginal='box',
                       title='Age Distribution vs Subscription Status',
                       barmode='overlay',
                       color_discrete_map={'yes':'#00CC96', 'no':'#EF553B'})
fig_age.show()



There are typical pattern from the client whicimportant to optimize the campaign efficiency in the future:
1. High rejection from clients, 88.3% rejected the offers and only 11.7% agreed. This shows that blind calling or calling everyone in the database is not an effective approach
2. Balance or average annual balance is an important key. Clients who chose to accept the offer have a higher median balance and IQR compared to those who rejected it. It means that clients with high liquidity tend to prefer using deposits.
3. The highest conversion is seen at the first call and remains stable at the second and third attempts, but decreases significantly in the next attempts. Therefore, the maximum attempts for a campaign should be set to three to avoid 'customer fatigue' and save on operational costs and time.
4. The campaign was mostly received by clients aged 30–40, but this also resulted in high rejection. On the other hand, clients who are students or retired have the highest conversion rates
5. Clients with personal and/or housing loans are much more likely to reject the offer. Existing financial commitments act as a major barrier, as these clients have less disposable income to lock into a term deposit.

## **Categorical Encoding**

In [82]:
binary_cols = ['default', 'housing', 'loan', 'y']
for col in binary_cols:
    df[col] = df[col].map({'yes': 1, 'no': 0})

df_encoded = pd.get_dummies(df, drop_first=True)

print(f"Original dataframe shape: {df.shape}")
print(f"Encoded dataframe shape: {df_encoded.shape}")
display(df_encoded.head(3))

Original dataframe shape: (45211, 17)
Encoded dataframe shape: (45211, 43)


,age,default,balance,housing,loan,day_of_month,duration,campaign,pdays,previous,...,month_jul,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_non_existent,poutcome_other,poutcome_success
0,58,0,2143,1,0,5,261,1.0,-1,0,...,False,False,False,True,False,False,False,True,False,False
1,44,0,29,1,0,5,151,1.0,-1,0,...,False,False,False,True,False,False,False,True,False,False
2,33,0,2,1,1,5,76,1.0,-1,0,...,False,False,False,True,False,False,False,True,False,False


In [83]:
from sklearn.model_selection import train_test_split

df_encoded = df_encoded.drop(columns=['duration'])
X = df_encoded.drop('y', axis=1)
y = df_encoded['y']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")

Training features shape: (36168, 41)
Testing features shape: (9043, 41)


In [84]:
from imblearn.over_sampling import SMOTE

print("Before SMOTE, training target distribution:")
print(y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("\nAfter SMOTE, training target distribution:")
print(y_train_smote.value_counts())

Before SMOTE, training target distribution:
y
0    31937
1     4231
Name: count, dtype: int64

After SMOTE, training target distribution:
y
0    31937
1    31937
Name: count, dtype: int64


## **Modeling**


In [85]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
xgb_model = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42, eval_metric='logloss')

rf_model.fit(X_train_smote, y_train_smote)
xgb_model.fit(X_train_smote, y_train_smote)

rf_pred = rf_model.predict(X_test)
xgb_pred = xgb_model.predict(X_test)

def get_report_df(y_true, y_pred, model_name):
    report = classification_report(y_true, y_pred, output_dict=True)
    df_report = pd.DataFrame(report).transpose()
    df_report.columns.name = model_name
    return df_report

display(get_report_df(y_test, rf_pred, "Random Forest Metrics"))
print('--')
display(get_report_df(y_test, xgb_pred, "XGBoost Metrics"))

Random Forest Metrics,precision,recall,f1-score,support
0,0.912078,0.966562,0.938530,7985.000000
1,0.540448,0.296786,0.383160,1058.000000
accuracy,0.888201,0.888201,0.888201,0.888201
macro avg,0.726263,0.631674,0.660845,9043.000000
weighted avg,0.868598,0.888201,0.873554,9043.000000


--


XGBoost Metrics,precision,recall,f1-score,support
0,0.912103,0.972073,0.941134,7985.000000
1,0.581614,0.293006,0.389692,1058.000000
accuracy,0.892624,0.892624,0.892624,0.892624
macro avg,0.746858,0.632539,0.665413,9043.000000
weighted avg,0.873437,0.892624,0.876617,9043.000000


In [86]:
cm_rf = confusion_matrix(y_test, rf_pred)
cm_xgb = confusion_matrix(y_test, xgb_pred)

labels = ['No (0)', 'Yes (1)']

fig_cm = make_subplots(rows=1, cols=2, subplot_titles=('Random Forest', 'XGBoost'))

fig_cm.add_trace(
    go.Heatmap(z=cm_rf, x=labels, y=labels, colorscale='Blues', showscale=False,
               text=cm_rf, texttemplate="%{text}", hoverinfo="z"),
    row=1, col=1
)

fig_cm.add_trace(
    go.Heatmap(z=cm_xgb, x=labels, y=labels, colorscale='Greens', showscale=False,
               text=cm_xgb, texttemplate="%{text}", hoverinfo="z"),
    row=1, col=2
)

fig_cm.update_layout(title_text='Confusion Matrix Comparison', height=500)
fig_cm.show()

In [87]:
def prepare_metrics_df(y_true, y_pred, model_name):
    report = classification_report(y_true, y_pred, output_dict=True)
    return {
        'Model': model_name,
        'Recall': report['1']['recall'],
        'F1-Score': report['1']['f1-score'],
        'Precision': report['1']['precision']
    }

metrics_data = [
    prepare_metrics_df(y_test, rf_pred, 'Random Forest'),
    prepare_metrics_df(y_test, xgb_pred, 'XGBoost')
]
df_metrics = pd.DataFrame(metrics_data)

fig_metrics = px.bar(df_metrics, x='Model', y=['Recall', 'F1-Score'],
                     barmode='group',
                     text_auto='.2f',
                     title='Model Performance Comparison (Class 1: Yes)',
                     labels={'value': 'Score', 'variable': 'Metric'},
                     color_discrete_sequence=['#636EFA', '#EF553B'])

fig_metrics.update_layout(yaxis=dict(range=[0, 1]))
fig_metrics.show()

In [88]:
importances = xgb_model.feature_importances_
feature_names = X_train_smote.columns

df_importance = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
df_importance = df_importance.sort_values(by='Importance', ascending=False).head(10)

fig_imp = px.bar(df_importance, x='Importance', y='Feature', orientation='h',
                 title='Top 10 Most Influential Features (XGBoost)',
                 labels={'Importance': 'Information Gain', 'Feature': 'Features'},
                 color='Importance', color_continuous_scale='Reds')

fig_imp.update_layout(yaxis={'categoryorder':'total ascending'})
fig_imp.show()

## **Simulation**

Asumption:

Cost per call: 1 USD

Profit per conversion (yes): 50 USD

Case A (Baseline): Blind campaign to all client in database.


Case B (Data-driven): Targeted campaign where only client classified as high probability

In [89]:
cost_per_call = 1
profit_per_yes = 50

total_calls_a = len(y_test)
total_yes_a = y_test.sum()
cost_a = total_calls_a * cost_per_call
revenue_a = total_yes_a * profit_per_yes
profit_a = revenue_a - cost_a
roi_a = (profit_a / cost_a)

total_calls_b = xgb_pred.sum()
tp_b = ((xgb_pred == 1) & (y_test == 1)).sum()
cost_b = total_calls_b * cost_per_call
revenue_b = tp_b * profit_per_yes
profit_b = revenue_b - cost_b
roi_b = (profit_b / cost_b)

summary_data = {
    'Metric': ['Total Calls', 'Total Conversion', 'Total Cost', 'Total Revenue', 'Net Profit', 'ROI'],
    'Massive Campaign': [
        f"{total_calls_a:,}", f"{total_yes_a:,}", f"${cost_a:,}", f"${revenue_a:,}", f"${profit_a:,}", f"{roi_a:.2f}x"
    ],
    'Data-Driven': [
        f"{total_calls_b:,}", f"{tp_b:,}", f"${cost_b:,}", f"${revenue_b:,}", f"${profit_b:,}", f"{roi_b:.2f}x"
    ]
}
df_summary = pd.DataFrame(summary_data)

display(df_summary.style.set_properties(**{'text-align': 'center'})
        .set_table_styles([{'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white')]}])
        .hide(axis='index'))

Metric,Massive Campaign,Data-Driven
Total Calls,"9,043",533
Total Conversion,"1,058",310
Total Cost,"$9,043",$533
Total Revenue,"$52,900","$15,500"
Net Profit,"$43,857","$14,967"
ROI,4.85x,28.08x


In [90]:
fig = make_subplots(
    rows=2, cols=2,
    specs=[[{"type": "indicator"}, {"type": "indicator"}],
           [{"type": "bar", "colspan": 2}, None]],
    vertical_spacing=0.3
)

fig.add_trace(go.Indicator(
    mode="number", value=roi_a,
    title={'text': "ROI: Massive Campaign"},
    number={'suffix': "x", 'font': {'color': '#EF553B'}},
    domain={'x': [0, 0.4], 'y': [0.6, 1]}
), row=1, col=1)

fig.add_trace(go.Indicator(
    mode="number", value=roi_b,
    title={'text': "ROI: Data-Driven"},
    number={'suffix': "x", 'font': {'color': '#00CC96'}},
    domain={'x': [0.6, 1], 'y': [0.6, 1]}
), row=1, col=2)

fig.add_trace(go.Bar(
    x=['Massive Campaign', 'Data-Driven'],
    y=[cost_a, cost_b],
    name='Total Cost', marker_color='#636EFA'
), row=2, col=1)

fig.add_trace(go.Bar(
    x=['Massive Campaign', 'Data-Driven'],
    y=[profit_a, profit_b],
    name='Net Profit', marker_color='#00CC96'
), row=2, col=1)

fig.update_layout(
    title_text="Financial Impact Simulation: Targeting Efficiency",
    height=700, showlegend=True, barmode='group'
)
fig.show()

## **Exporting data for Tableau**

In [91]:
df_tableau = df.loc[X_test.index].copy()

In [92]:
df_tableau['Model_Prediction'] = xgb_pred
df_tableau['Model_Prediction_Label'] = df_tableau['Model_Prediction'].map({1: 'Predicted Yes', 0: 'Predicted No'})
df_tableau.head()

,age,job,marital,education,default,balance,housing,loan,contact,day_of_month,month,duration,campaign,pdays,previous,poutcome,y,Model_Prediction,Model_Prediction_Label
1392,40,blue-collar,married,primary,0,640,1,1,unknown,8,may,347,2.0,-1,0,non_existent,0,0,Predicted No
7518,44,technician,married,secondary,0,378,1,0,unknown,30,may,203,2.0,-1,0,non_existent,0,0,Predicted No
12007,31,services,married,secondary,0,356,1,0,unknown,20,jun,228,5.0,-1,0,non_existent,0,0,Predicted No
5536,36,blue-collar,married,primary,0,655,1,0,unknown,23,may,153,4.0,-1,0,non_existent,0,0,Predicted No
29816,34,services,single,secondary,0,1921,1,0,cellular,4,feb,61,1.0,-1,0,non_existent,0,0,Predicted No


In [93]:
df_tableau['Probability_Score'] = xgb_model.predict_proba(X_test)[:, 1]

df_tableau['Actual_Label'] = y_test

df_tableau.head()

,age,job,marital,education,default,balance,housing,loan,contact,day_of_month,...,duration,campaign,pdays,previous,poutcome,y,Model_Prediction,Model_Prediction_Label,Probability_Score,Actual_Label
1392,40,blue-collar,married,primary,0,640,1,1,unknown,8,...,347,2.0,-1,0,non_existent,0,0,Predicted No,0.040486,0
7518,44,technician,married,secondary,0,378,1,0,unknown,30,...,203,2.0,-1,0,non_existent,0,0,Predicted No,0.061974,0
12007,31,services,married,secondary,0,356,1,0,unknown,20,...,228,5.0,-1,0,non_existent,0,0,Predicted No,0.053469,0
5536,36,blue-collar,married,primary,0,655,1,0,unknown,23,...,153,4.0,-1,0,non_existent,0,0,Predicted No,0.042169,0
29816,34,services,single,secondary,0,1921,1,0,cellular,4,...,61,1.0,-1,0,non_existent,0,0,Predicted No,0.119928,0


In [94]:
df_tableau.to_csv('bank_marketing_with_label.csv', index=False)